## Imports

In [2]:
import pandas as pd
import numpy as np
from numpy import genfromtxt
import zipfile as zipfile

import sys
import pickle
import requests

## Global variables

In [3]:
# URL of datasources
url_mdl                = 'https://files.grouplens.org/datasets/movielens/ml-20m.zip'

url_imdb_namebasics    = 'https://datasets.imdbws.com/name.basics.tsv.gz'
url_imdb_titleakas     = 'https://datasets.imdbws.com/title.akas.tsv.gz'
url_imdb_titlecrew     = 'https://datasets.imdbws.com/title.crew.tsv.gz'
url_imdb_titlebasics   = 'https://datasets.imdbws.com/title.basics.tsv.gz'
url_imdb_titleepisode  = 'https://datasets.imdbws.com/title.episode.tsv.gz'
url_imdb_titleprincipals  = 'https://datasets.imdbws.com/title.principals.tsv.gz'
url_imdb_titleratings  = 'https://datasets.imdbws.com/title.ratings.tsv.gz'

## Pre-process

In [6]:
# Load movie dataset
movies = pd.read_csv('data_mdl/movies.zip',compression='zip', sep=',')

# Limitation of dataset using pre-defined list
list_movies = np.genfromtxt('data_processed/list_movieId_2000_2010.csv',delimiter=",", dtype=int)
movies = movies[movies['movieId'].isin(list_movies)]

# Split year from title
movies['year'] = movies['title'].str.extract('.*\((.*)\).*',expand = False)
movies['title'] = movies['title'].apply(lambda x: x[:-7])

# Clean of years values
dict_replace = {'1975-1979': '1979',
'2009– ': '2009',
'2007-':'2007', 
'Das Millionenspiel':'1970', 
'Bicicleta, cullera, poma':'2010',
'1983)':'1983'
}

movies['year'] = movies['year'].replace(to_replace=dict_replace) 

# Drop of rows with missing year (17 movies)
movies_without_year = movies[movies['year'].isnull()][['movieId', 'title']]
movies = movies.dropna(axis=0, how='any', subset=['year'])

# Change 'year' type for int
movies['year'] = movies['year'].astype('int')

# Clean of genres column
movies['genres'] = movies['genres'].str.replace('|',' ')


print('Nbre de films : ', len(movies.movieId.unique()), '\n')
print(movies.info())
movies.head(5)

Nbre de films :  7264 

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7264 entries, 3958 to 27276
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  7264 non-null   int64 
 1   title    7264 non-null   object
 2   genres   7264 non-null   object
 3   year     7264 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 283.8+ KB
None


/tmp/ipykernel_24579/2237409829.py:36: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  movies['genres'] = movies['genres'].str.replace('|',' ')


,movieId,title,genres,year
3958,4052,Antitrust,Crime Drama Thriller,2001
3959,4053,Double Take,Action Comedy,2001
3960,4054,Save the Last Dance,Drama Romance,2001
3962,4056,"Pledge, The",Crime Drama Mystery Thriller,2001
3974,4068,Sugar & Spice,Comedy,2001


In [7]:
# Load /  A ajouter : chargement direct dpuis url
links = pd.read_csv('data_mdl/links.zip',compression='zip', sep=',')

links = links.drop(['tmdbId'], axis=1)

print('Nbre de films : ', len(movies.movieId.unique()), '\n')
print('imdbId min : ',links.imdbId.min(), ' imdbId max : ', links.imdbId.max(), '\n')
print(links.info())
links.head(5)

Nbre de films :  7264 

imdbId min :  5  imdbId max :  4530184 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  27278 non-null  int64
 1   imdbId   27278 non-null  int64
dtypes: int64(2)
memory usage: 426.3 KB
None


,movieId,imdbId
0,1,114709
1,2,113497
2,3,113228
3,4,114885
4,5,113041


## Merge

In [9]:
mdl_content = movies.merge(right=links, left_on='movieId', right_on='movieId', how='left')


mdl_content.head(5)

,movieId,title,genres,year,imdbId
0,4052,Antitrust,Crime Drama Thriller,2001,218817
1,4053,Double Take,Action Comedy,2001,238948
2,4054,Save the Last Dance,Drama Romance,2001,206275
3,4056,"Pledge, The",Crime Drama Mystery Thriller,2001,237572
4,4068,Sugar & Spice,Comedy,2001,186589


## Save

In [10]:
mdl_content.to_csv("data_processed/mdl_content.csv.zip", index=False, compression="zip")